# CSV 정형 데이터를 지식 그래프로 변환

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE")
)

In [ ]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph()

In [ ]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

# 그래프 DB 설계 명세

## 1. 설계 개요

본 그래프 DB는 영화 정보를 중심으로 영화, 인물, 장르 간의 관계를 표현하기 위한 구조이다.
영화는 `Movie` 노드로 표현하고, 감독과 배우는 `Person` 노드로 표현한다. 장르는 `Genre` 노드로 표현한다.
영화와 인물 사이에는 감독 관계와 출연 관계를 정의하고, 영화와 장르 사이에는 장르 분류 관계를 정의한다.

## 2. 노드 설계

### 2.1 Movie 노드

`Movie` 노드는 영화 정보를 표현하는 노드이다.

| 항목    | 내용                              |
| ----- | ----------------------------------- |
| Label | `Movie`                             |
| 설명    | 영화 정보를 저장하는 노드         |
| 속성 | `id`, `title`, `released`, `rating` |

#### 속성 정의

| 속성명        | 타입                    | 설명           | 예                            |
| ---------- | --------------------- | ------------ | ---------------------------- |
| `id`       | `STRING`             | 영화의 고유 식별자   | `"movie-1000"` |
| `title`    | `STRING`              | 영화 제목        | `"The Matrix"`               |
| `released` | `INTEGER` 또는 `DATE` | 개봉 연도 또는 개봉일 | `1999`, `date("1999-03-31")` |
| `rating`   | `FLOAT`               | 영화 평점        | `8.7`                        |

### 2.2 Person 노드

`Person` 노드는 영화와 관련된 인물을 표현하는 노드이다.
감독과 배우를 별도의 라벨로 분리하지 않고, 모두 `Person` 라벨로 관리한다.

| 항목  | 내용                                    |
| ----- | --------------------------------------- |
| Label | `Person`                                |
| 설명  | 감독 또는 배우 정보를 저장하는 노드     |
| 속성  | `name`                                |

#### 속성 정의

| 속성명    | 타입  | 설명  | 예                                        |
| ------ | -------- | ----- | ----------------------------------------- |
| `name` | `STRING` | 인물 이름 | `"Lana Wachowski"`, `"Keanu Reeves"`  |

### 2.3 Genre 노드

`Genre` 노드는 영화의 장르 정보를 표현하는 노드이다.

| 항목    | 내용                           |
| ----- | -------------------------------- |
| Label | `Genre`                          |
| 설명  | 영화 장르 정보를 저장하는 노드 |
| 속성  | `name`                         |

#### 속성 정의

| 속성명    | 타입  | 설명   | 예                 |
| ------ | -------- | ------ | ------------------ |
| `name` | `STRING` | 장르명 | `"Action"`, `"SF"` |

## 3. 관계 설계

### 3.1 DIRECTED 관계

`DIRECTED`: 인물이 영화를 감독했음을 표현한다.

| 항목              | 내용                             |
| ----------------- | -------------------------------- |
| Relationship Type | `DIRECTED`                       |
| 시작 노드         | `Person`                         |
| 종료 노드         | `Movie`                          |
| 방향              | `Person` → `Movie`               |
| 의미              | 특정 인물이 특정 영화를 감독했다 |

```cypher
(:Person)-[:DIRECTED]->(:Movie)
```

### 3.2 ACTED_IN 관계

`ACTED_IN`: 인물이 영화에 출연했음을 표현한다.

| 항목              | 내용                                    |
| ----------------- | --------------------------------------- |
| Relationship Type | `ACTED_IN`                              |
| 시작 노드         | `Person`                                |
| 종료 노드         | `Movie`                                 |
| 방향              | `Person` → `Movie`                      |
| 의미              | 특정 인물이 특정 영화에 배우로 출연했다 |

```cypher
(:Person)-[:ACTED_IN]->(:Movie)
```

### 3.3 IN_GENRE 관계

`IN_GENRE`: 영화가 특정 장르에 속함을 표현한다.

| 항목              | 내용                           |
| ----------------- | ------------------------------ |
| Relationship Type | `IN_GENRE`                     |
| 시작 노드         | `Movie`                        |
| 종료 노드         | `Genre`                        |
| 방향              | `Movie` → `Genre`              |
| 의미              | 특정 영화가 특정 장르에 속한다 |

```cypher
(:Movie)-[:IN_GENRE]->(:Genre)
```

## 5. 제약조건 설계

데이터의 중복을 방지하고 필수 속성을 보장하기 위해 다음 제약조건을 설정한다.

### 5.1 Movie 노드 제약조건

`Movie` 노드의 `id`는 영화의 고유 식별자로 사용한다. 따라서 중복되지 않아야 한다.

```cypher
CREATE CONSTRAINT movie_id_unique IF NOT EXISTS
FOR (m:Movie)
REQUIRE m.id IS UNIQUE;
```

`Movie` 노드는 반드시 `title` 속성을 가져야 한다.

```cypher
CREATE CONSTRAINT movie_title_exists IF NOT EXISTS
FOR (m:Movie)
REQUIRE m.title IS NOT NULL;
```

### 5.2 Person 노드 제약조건

`Person` 노드의 `name`은 인물을 식별하는 주요 속성이다. 동일 이름 인물을 구분하지 않기 때문에 `name`을 유일한 값으로 관리할 수 있다.

```cypher
CREATE CONSTRAINT person_name_unique IF NOT EXISTS
FOR (p:Person)
REQUIRE p.name IS UNIQUE;
```

### 5.3 Genre 노드 제약조건

`Genre` 노드의 `name`은 장르를 식별하는 값이므로 중복되지 않도록 한다.

```cypher
CREATE CONSTRAINT genre_name_unique IF NOT EXISTS
FOR (g:Genre)
REQUIRE g.name IS UNIQUE;
```

## 6. 인덱스 설계

자주 검색하는 속성에는 인덱스를 생성한다.

### 6.1 영화 제목 검색 인덱스

영화 제목으로 검색하는 경우가 많으므로 `Movie.title`에 인덱스를 생성한다.

```cypher
CREATE INDEX movie_title_index IF NOT EXISTS
FOR (m:Movie)
ON (m.title);
```

### 6.2 영화 평점 검색 인덱스

평점 기준으로 영화를 조회하거나 정렬하는 경우를 고려하여 `Movie.rating`에 인덱스를 생성한다.

```cypher
CREATE INDEX movie_rating_index IF NOT EXISTS
FOR (m:Movie)
ON (m.rating);
```

### 6.3 개봉 연도 검색 인덱스

개봉 연도 또는 개봉일 기준으로 영화를 조회할 수 있으므로 `Movie.released`에 인덱스를 생성한다.

```cypher
CREATE INDEX movie_released_index IF NOT EXISTS
FOR (m:Movie)
ON (m.released);
```

- 생성된 노드, 속성 정보 조회
  - `CALL db.schema.visualization() ` 
    - Label과 Relationship 구조를 시작적으로 표시
  - `CALL db.schema.nodeTypeProperties()` 
    - 노드의 속성 조회
  - `CALL db.schema.relTypeProperties()` 
    - 관계의 속성 조회



In [ ]:
###################################################
# 제약조건 구문
###################################################
constraints = [
    "CREATE CONSTRAINT movie_id_unique IF NOT EXISTS FOR (m:Movie) REQUIRE m.id IS UNIQUE",
    "CREATE CONSTRAINT movie_title_exists IF NOT EXISTS FOR (m:Movie) REQUIRE m.title IS NOT NULL",
    "CREATE CONSTRAINT person_name_unique IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT genre_name_unique IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE"
]

###################################
# 인덱스 생성 구문 
###################################
indexes = [
    "CREATE INDEX movie_title_index IF NOT EXISTS FOR (m:Movie) ON (m.title)",
    "CREATE INDEX movie_rating_index IF NOT EXISTS FOR (m:Movie) ON (m.rating)",
    "CREATE INDEX movie_release_index IF NOT EXISTS FOR (m:Movie) ON (m.released)",
]


for constraint in constraints:
    graph.query(constraint)
for index in indexes:
    graph.query(index)

print("스키마 설정 완료")

In [ ]:
# 제약 조건 조회
graph.query("show constraints")

In [ ]:
# index 조회
graph.query("SHOW INDEXES")

### 2.2 CSV 데이터로 지식 그래프 구축

- CSV 형태의 영화 데이터셋을 읽어서 그래프 구조로 변환
- 영화, 인물(감독, 배우), 장르를 노드로 변환
- 각 노드 간의 관계(DIRECTED, ACTED_IN, IN_GENRE)를 생성
- 변환된 데이터를 Neo4j 그래프 데이터베이스에 저장

- **CSV 파일 로드**:

    - **TMDB**와 **GroupLens**에서 수집한 데이터를 결합하여 정리
    - 영화 상세정보, 제작진 정보, 키워드는 **TMDB Open API**를 통해 수집됨
    - 영화 링크와 평점 정보는 **공식 GroupLens 웹사이트**에서 획득함
    - 출처: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system

In [ ]:
import pandas as pd
df = pd.read_csv('data/movies_tmdb_small.csv')
print(df.shape)

df.head(5)

- **필수 라이브러리**:
    - Neo4j 그래프 데이터베이스와 연동하여 문서 처리 및 데이터 분석을 위한 기본 설정

In [ ]:
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

- **데이터 변환**: 
    - Langchain-neo4j의 **Node, Relationship 클래스**를 이용해 Node와 Relationship을 정의한다.
  
    1. **Node**: 그래프의 노드(Node)를 표현
       - `id`: 노드의 고유 식별자
       - `type`: 노드의 유형(예: Movie, Person, Genre)
       - `properties`: 노드의 속성들을 dict로 설정

    2. **Relationship**: 두 노드 간의 관계(Relationship)를 표현
       - `source`: 관계의 시작 노드 (Person)
       - `target`: 관계의 목표 노드 (Movie)
       - `type`: 관계의 유형(예: DIRECTED)
       - `properties`: 관계의 속성들


In [ ]:
##########################################################################
# DataFrame의 내용을 그래프 구조로 변환하기 위해 노드와 관계를 생성한다.
##########################################################################

# 노드 ID를 키로 사용하여 생성된 노드 객체를 저장할 dict
node_dict = {}  

# 관계를 저장할 리스트
relationships = []

batch_size = 100

total_rows = len(df)
for batch_start in range(0, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    batch_df = df.iloc[batch_start:batch_end]
    
    ####################################################### 
    # 배치 내 CSV 데이터를 순회하며 그래프 구조로 변환
    #######################################################
    for _, row in batch_df.iterrows():

        # 영화 정보 처리
        movie_id = f"movie-{row['id']}"
        if movie_id not in node_dict:

            movie_properties = {
                "id": movie_id,
                "title": row['title'],
                "released": row['released'],
                "rating": float(row['rating']) if pd.notna(row['rating']) else None
            }
            
            if pd.notna(row.get('overview')):
                movie_properties["overview"] = row['overview']
            
            if pd.notna(row.get('runtime')):
                try:
                    movie_properties["runtime"] = int(row['runtime'])
                except (ValueError, TypeError):
                    movie_properties["runtime"] = row['runtime']
            
            if pd.notna(row.get('tagline')):
                movie_properties["tagline"] = row['tagline']
            
            movie_node = Node(
                id=movie_id,
                type="Movie",
                properties=movie_properties
            )
            
            node_dict[movie_id] = movie_node
        
        # 감독 정보 처리 (결측값이 아닌 경우에만)
        if pd.notna(row.get('director')):
            for director in row['director'].split('|'):
                director = director.strip()
                director_id = f"person-{director}"
                
                if director_id not in node_dict:
                    director_node = Node(
                        id=director_id,
                        type="Person",
                        properties={"name": director}
                    )
                    
                    node_dict[director_id] = director_node
                
                relationships.append(
                    Relationship(
                        source=node_dict[director_id],
                        target=node_dict[movie_id],
                        type="DIRECTED",
                        properties={}
                    )
                )
        
        # 배우 정보 처리 (결측값이 아닌 경우에만)
        if pd.notna(row.get('actors')):
            
            for actor in row['actors'].split('|'):
                actor = actor.strip() 
                actor_id = f"person-{actor}"  
                                
                if actor_id not in node_dict:
                    actor_node = Node(
                        id=actor_id,
                        type="Person",  
                        properties={"name": actor}  
                    )
                    
                    node_dict[actor_id] = actor_node
                
                relationships.append(
                    Relationship(
                        source=node_dict[actor_id],
                        target=node_dict[movie_id],
                        type="ACTED_IN",
                        properties={}
                    )
                )
        
        # 장르 정보 처리 (결측값이 아닌 경우에만)
        if pd.notna(row.get('genres')):
            for genre in row['genres'].split('|'):
                genre = genre.strip() 
                genre_id = f"genre-{genre}"
                
                
                if genre_id not in node_dict:
                    genre_node = Node(
                        id=genre_id,
                        type="Genre",
                        properties={"name": genre}
                    )
                   
                    node_dict[genre_id] = genre_node
                
                
                relationships.append(
                    Relationship(
                        source=node_dict[movie_id],
                        target=node_dict[genre_id],
                        type="IN_GENRE",
                        properties={}
                    )
                )
    
    print(f"배치 처리 완료: {batch_start+1}~{batch_end}/{total_rows} 레코드")

# 결과 출력
print(f"총 노드 수: {len(node_dict)}")
print(f"총 관계 수: {len(relationships)}")

In [ ]:
node_dict.keys()
for i, (k, v) in enumerate(node_dict.items()):
    print(k, v, sep="::")
    if i == 2:
        break

In [ ]:
relationships[:2]

- **GraphDocument 생성 및 저장**:

    - **GraphDocument**: 그래프 전체 문서를 표현
        - `nodes`: 모든 노드의 리스트
        - `relationships`: 모든 관계의 리스트

    - 모든 노드와 관계를 수집한 후, `GraphDocument` 객체를 생성하고 이를 그래프 데이터베이스에 저장

In [ ]:
nodes = list(node_dict.values())

# GraphDocument 객체 생성
graph_doc = GraphDocument(
    nodes=nodes,
    relationships=relationships
)

# 생성된 GraphDocument를 Neo4j 데이터베이스에 저장
# add_graph_documents 메서드는 GraphDocument 객체 리스트를 받아 데이터베이스에 일괄 저장
graph.add_graph_documents([graph_doc])
print("그래프 데이터베이스에 저장 완료")

In [ ]:
# 노드 개수 확인
graph.query(
    """MATCH (n:Movie)
WITH count(n) AS 영화수
MATCH (p:Person)
WITH 영화수, count(p) AS 사람수
MATCH (g:Genre)
RETURN 영화수, 사람수, count(g) AS 장르수"""
)